In [ ]:
pip install openai typing_extensions langgraph


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


UnboundLocalError: cannot access local variable 'child' where it is not associated with a value

In [ ]:
from openai import OpenAI
APIKEY = "sxxxxx"
client = OpenAI(api_key=APIKEY)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "introduce yourself"}]
)

print(response.choices[0].message.content)

Hello! I'm an AI language model created by OpenAI, designed to assist with a variety of questions and tasks, providing information, generating text, and engaging in conversation. Whether you need help with research, writing, or simply want to chat, I'm here to help! What can I assist you with today?


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=APIKEY)

ARTICLE_PROMPT = """ 
You are an expert in writing English learning materials.
Please write an article suitable for English learners at the {level} level based on the given topic.

Requirements:

* The length should be between 200 and 300 words
* Vocabulary and sentence structures must strictly match the {level} level (CEFR standard)
* The content should be interesting and close to {topic}
* Output only the main body of the English article, without a title, without any Chinese explanations, and without markdown formatting
"""

def generate_article(topic: str, level: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": ARTICLE_PROMPT.format(topic=topic, level=level)
            }
        ]
    )
    return response.choices[0].message.content.strip()

# test
if __name__ == "__main__":
    article = generate_article("order a cup of coffee", "A2")
    print(article)
    print(len(article.split()))


Many people love to drink coffee. It is a popular drink all over the world. Coffee helps us wake up in the morning and feel good. There are many different types of coffee, and each type has its own taste.

One of the most common types of coffee is black coffee. This coffee is simple. It has coffee and hot water only. Some people like black coffee because it has a strong flavor. Others like to add milk or sugar to make it sweeter. This is called a latte or a cappuccino. A latte has more milk, while a cappuccino has more foam.

When you go to a coffee shop, you can see many choices. You can order a small cup or a large one. Some shops have special coffees, like vanilla or caramel flavors. These coffees can taste very nice and sweet. 

If you want to make coffee at home, it is easy. You need coffee beans or coffee powder, hot water, and a coffee maker. First, you put the coffee in the maker. Then, you add hot water. After a few minutes, your coffee is ready! You can drink it black or add 

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import json


# ① define the structure of output
class DifficultyAssessment(BaseModel):
    is_appropriate: bool      
    actual_level: str          
    feedback: str              

ASSESS_PROMPT = """
You are an expert in assessing English proficiency based on the CEFR standard.
Please evaluate whether the following article matches the {level} level.

Evaluation criteria:

* Whether the vocabulary falls within the commonly used range for this level
* Whether the sentence complexity (clauses, tenses) is appropriate
* Whether the article length is reasonable

Article content:
{article}

Please output the evaluation result in the specified JSON format.
"""

def assess_difficulty(article: str, level: str) -> DifficultyAssessment:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": ASSESS_PROMPT.format(level=level, article=article)}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "difficulty_assessment",
                "schema": {
                    "type": "object",
                    "properties": {
                        "is_appropriate": {"type": "boolean"},
                        "actual_level": {"type": "string"},
                        "feedback": {"type": "string"}
                    },
                    "required": ["is_appropriate", "actual_level", "feedback"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )
    result = json.loads(response.choices[0].message.content)
    return DifficultyAssessment(**result)


# ② revise_article node
REVISE_PROMPT = """
You are an expert in writing English learning materials.
Please revise the following article based on the feedback so that it better matches the {level} level.

Original text:
{article}

Feedback:
{feedback}

Requirements:

* Output only the revised English article body
* Do not include any explanation
* Keep the length between 200 and 300 words
"""

def revise_article(article: str, level: str, feedback: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": REVISE_PROMPT.format(level=level, article=article, feedback=feedback)}
        ]
    )
    return response.choices[0].message.content


# ③ test the whole process
if __name__ == "__main__":
    article = """
        Many people love to drink coffee. Coffee is a popular drink around the world. It gives us energy and makes us feel good. There are many ways to prepare coffee. You can drink it black or with milk. Many people like to add sugar or flavor. 

        In a coffee shop, you can find different types of coffee. The most common are espresso, cappuccino, and latte. Espresso is a strong and small cup of coffee. Cappuccino has equal parts of espresso, hot milk, and foam. Latte has more milk and less foam than cappuccino. These drinks are very tasty!

        When you order coffee, you can say, “I would like a coffee, please.” If you want a special drink, you can add the name. For example, “I would like a cappuccino, please.” You can also choose the size. A small coffee is called a "short," and a large coffee is called a "tall."

        Do not forget to choose your favorite flavor! Some coffee shops have vanilla, caramel, or hazelnut. You can say, “I want a vanilla latte, please.” 

        Many people enjoy coffee with friends. It is nice to sit together and talk. Coffee shops are good places for meetings and study too. 

        So, the next time you want a break or a nice drink, think about ordering a coffee. It can make your day better!

    """ 

    level = "A2"

    assessment = assess_difficulty(article, level)
    print("evaluation-1：")
    print(f"  is ok：{assessment.is_appropriate}")
    print(f"  real level：{assessment.actual_level}")
    print(f"  feedback：{assessment.feedback}")

    if not assessment.is_appropriate:
        print("\nstart revise article...")
        revised = revise_article(article, level, assessment.feedback)
        print(revised)
    
    hard_article = """
        The proliferation of artificial intelligence has precipitated a paradigm shift in contemporary discourse, engendering both unprecedented opportunities and formidable challenges. Proponents argue that AI's capacity for autonomous decision-making could substantially augment human productivity, particularly in sectors characterized by repetitive, algorithmically-tractable tasks.

        Conversely, skeptics contend that the unmitigated proliferation of such technologies risks exacerbating socioeconomic disparities, insofar as the displacement of labor disproportionately affects individuals lacking access to retraining initiatives. Furthermore, the opacity inherent in many machine learning architectures—colloquially termed the "black box" phenomenon—raises pressing questions regarding accountability and transparency.

        Notwithstanding these concerns, a growing consensus suggests that the judicious implementation of regulatory frameworks could mitigate potential harms while preserving the innovative momentum that has characterized this domain. Whether such frameworks can be operationalized expeditiously, however, remains a matter of considerable debate among policymakers and industry stakeholders alike.
    """
    level = "A2"

    assessment = assess_difficulty(hard_article, level)
    print("evaluation-2：")
    print(f"  is ok：{assessment.is_appropriate}")
    print(f"  real level：{assessment.actual_level}")
    print(f"  feedback：{assessment.feedback}")

    if not assessment.is_appropriate:
        print("\nstart revise article....")
        revised = revise_article(hard_article, level, assessment.feedback)
        print(revised)
        print(f"\n after revise：{len(revised.split())}")
        
        # evaluate the revised article
        re_assessment = assess_difficulty(revised, level)
        print(f"\n revise is ok?{re_assessment.is_appropriate}")
        print(f"after revise, level：{re_assessment.actual_level}")

evaluation-1：
  is ok：True
  real level：A2
  feedback：The article uses basic vocabulary related to coffee, which is commonly understood at the A2 level. The sentence structures are simple and mainly consist of short, clear sentences. The length of the article is reasonable for A2 learners, providing enough content without becoming overwhelming.
evaluation-2：
  is ok：False
  real level：C2
  feedback：The article contains advanced vocabulary (e.g., 'proliferation', 'paradigm shift', 'autonomous decision-making') that is well above the A2 level. The sentences are complex and contain multiple clauses, which is not suitable for A2 proficiency. The length of the article is also quite long, which is typically not expected at the A2 level.

start revise article....
Artificial intelligence (AI) is becoming more common in our lives. It can help us do many tasks faster and more efficiently. Many people believe that AI can make our work easier, especially in jobs that need repeating the same steps 

In [ ]:
def generate_with_revision_loop(article: str, level: str, max_revisions: int = 3):
    revision_count = 0

    while True:
        assessment = assess_difficulty(article, level)
        print(f"The {revision_count} time evaluation → ok：{assessment.is_appropriate}，level：{assessment.actual_level}")

        if assessment.is_appropriate:
            print("✅ value is appropriate, end loop")
            break

        revision_count += 1

        if revision_count > max_revisions:
            print(f"⚠️ reach the maximum revise time ({max_revisions})，force end loop")
            break

        print(f"the {revision_count} time revise article...")
        article = revise_article(article, level, assessment.feedback)

    return article, revision_count


# test
final_article, count = generate_with_revision_loop(hard_article, "A2", max_revisions=3)
print(f"\nthe final revise times：{count}")
print(f"final article：\n{final_article}")

The 0 time evaluation → ok：False，level：C1
the 1 time revise article...
The 1 time evaluation → ok：False，level：B1
the 2 time revise article...
The 2 time evaluation → ok：False，level：B2
the 3 time revise article...
The 3 time evaluation → ok：False，level：B1
⚠️ reach the maximum revise time (3)，force end loop

the final revise times：4
final article：
The rise of artificial intelligence (AI) is changing our world in many ways. AI gives us new opportunities, but it also brings some problems. Many people believe that AI can help us with our jobs, especially with tasks that are simple and repeated. For example, AI can do some work faster and make fewer mistakes than humans.

However, some people are worried about AI. They think that using AI too fast might hurt jobs. This is especially true for workers who cannot learn new skills. If machines take over many jobs, it could create bigger gaps in wealth and opportunities among people.

Another concern is that some AI systems are hard to understand

In [ ]:
class VocabularyItem(BaseModel):
    word: str           
    meaning: str        
    example: str        

class VocabularyList(BaseModel):
    words: list[VocabularyItem]

VOCAB_PROMPT = """
    You are an expert in teaching English vocabulary.
    Please select 5–8 words or phrases worth learning from the following {level}-level article.

    Selection criteria:

    * Prioritize words that are slightly above the learner’s current level but still learnable with effort
    * Avoid overly simple words (such as “the”, “is”, “go”)
    * For each word, provide its Chinese meaning and an example sentence from the article

    Article:
    {article}

    Please output the results in the specified JSON format.
"""

def extract_vocabulary(article: str, level: str) -> VocabularyList:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": VOCAB_PROMPT.format(level=level, article=article)}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "vocabulary_list",
                "schema": {
                    "type": "object",
                    "properties": {
                        "words": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "word": {"type": "string"},
                                    "meaning": {"type": "string"},
                                    "example": {"type": "string"}
                                },
                                "required": ["word", "meaning", "example"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["words"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )
    result = json.loads(response.choices[0].message.content)
    return VocabularyList(**result)


# test
if __name__ == "__main__":
    vocab = extract_vocabulary(article, level)
    for item in vocab.words:
        print(f"{item.word} — {item.meaning}")
        print(f"  example：{item.example}\n")

espresso — 一种强烈且少量的咖啡。
  example：The most common are espresso, cappuccino, and latte.

cappuccino — 由等量的浓缩咖啡、热牛奶和奶泡组成的咖啡。
  example：Cappuccino has equal parts of espresso, hot milk, and foam.

latte — 比卡布奇诺更多牛奶的咖啡。
  example：Latte has more milk and less foam than cappuccino.

flavor — 饮料中添加的味道，通常指各种你喜欢的。
  example：Do not forget to choose your favorite flavor!

tasty — 美味的，味道好的。
  example：These drinks are very tasty!

order — 请求食物或饮料的过程。
  example：When you order coffee, you can say, 'I would like a coffee, please.'

caramel — 一种甜味，通常由糖加热后制成的调味品。
  example：Some coffee shops have vanilla, caramel, or hazelnut.

hazelnut — 一种坚果，与咖啡搭配时常被用作风味。
  example：Some coffee shops have vanilla, caramel, or hazelnut.



In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel
import operator

class VocabularyItem(BaseModel):
    word: str
    meaning: str
    example: str

class VocabularyList(BaseModel):
    words: list[VocabularyItem]

class EchoReadState(TypedDict):
    topic: str
    level: str
    article: str
    feedback: str
    is_appropriate: bool
    revision_count: int
    max_revisions: int
    vocabulary: list

In [ ]:
from openai import OpenAI
from langgraph.graph import StateGraph, END
from pydantic import BaseModel
import json

client = OpenAI(api_key=APIKEY)

# ============ Prompts  ============

ARTICLE_PROMPT = """
    You are an expert in writing English learning materials.
    Please write an article suitable for English learners at the {level} level based on the given topic.

    Requirements:

    * The length should be between 200 and 300 words
    * Vocabulary and sentence structures must strictly match the {level} level (CEFR standard)
    * The content should be interesting and close to real life
    * Output only the main body of the English article, without a title, without any Chinese explanations, and without markdown formatting
"""

ASSESS_PROMPT = """
    You are an expert in assessing English proficiency based on the CEFR standard.
    Please evaluate whether the following article matches the {level} level.

    Evaluation criteria:

    * Whether the vocabulary falls within the commonly used range for this level
    * Whether the sentence complexity (clauses, tenses) is appropriate
    * Whether the article length is reasonable

    Article content:
    {article}

    Please output the evaluation result in the specified JSON format.
"""

REVISE_PROMPT = """
    You are an expert in writing English learning materials.
    Please revise the following article based on the feedback so that it better matches the {level} level.

    Original text:
    {article}

    Feedback:
    {feedback}

    Requirements:

    * Output only the revised English article body
    * Do not include any explanation
    * Keep the length between 200 and 300 words
"""

VOCAB_PROMPT = """
    You are an expert in teaching English vocabulary.
    Please select 5–8 words or phrases worth learning from the following {level}-level article.

    Selection criteria:

    * Prioritize words that are slightly above the learner’s current level but still learnable with effort
    * Avoid overly simple words
    * For each word, provide its Chinese meaning and an example sentence from the article

    Article:
    {article}

    Please output the results in the specified JSON format.
"""

# ============ Nodes ============

def generate_article_node(state: EchoReadState):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": ARTICLE_PROMPT.format(level=state["level"]) + f"\n\n topic:{state['topic']}"
        }]
    )
    return {"article": response.choices[0].message.content}


def assess_difficulty_node(state: EchoReadState):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": ASSESS_PROMPT.format(level=state["level"], article=state["article"])
        }],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "difficulty_assessment",
                "schema": {
                    "type": "object",
                    "properties": {
                        "is_appropriate": {"type": "boolean"},
                        "actual_level": {"type": "string"},
                        "feedback": {"type": "string"}
                    },
                    "required": ["is_appropriate", "actual_level", "feedback"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )
    result = json.loads(response.choices[0].message.content)
    return {
        "is_appropriate": result["is_appropriate"],
        "feedback": result["feedback"]
    }


def revise_article_node(state: EchoReadState):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": REVISE_PROMPT.format(
                level=state["level"],
                article=state["article"],
                feedback=state["feedback"]
            )
        }]
    )
    return {
        "article": response.choices[0].message.content,
        "revision_count": state["revision_count"] + 1
    }


def extract_vocabulary_node(state: EchoReadState):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": VOCAB_PROMPT.format(level=state["level"], article=state["article"])
        }],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "vocabulary_list",
                "schema": {
                    "type": "object",
                    "properties": {
                        "words": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "word": {"type": "string"},
                                    "meaning": {"type": "string"},
                                    "example": {"type": "string"}
                                },
                                "required": ["word", "meaning", "example"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["words"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )
    result = json.loads(response.choices[0].message.content)
    return {"vocabulary": result["words"]}


# ============ conditional_edges = ============

def should_revise(state: EchoReadState):
    if state["is_appropriate"]:
        return "vocabulary"
    if state["revision_count"] >= state["max_revisions"]:
        print(f"⚠️ reached the maximum revision time({state['max_revisions']}), enforcing next step")
        return "vocabulary"
    return "revise"


# ============ graph ============

graph_builder = StateGraph(EchoReadState)

graph_builder.add_node("generate", generate_article_node)
graph_builder.add_node("assess", assess_difficulty_node)
graph_builder.add_node("revise", revise_article_node)
graph_builder.add_node("vocabulary", extract_vocabulary_node)

graph_builder.set_entry_point("generate")
graph_builder.add_edge("generate", "assess")

graph_builder.add_conditional_edges(
    "assess",
    should_revise,
    {
        "revise": "revise",
        "vocabulary": "vocabulary"
    }
)

graph_builder.add_edge("revise", "assess")
graph_builder.add_edge("vocabulary", END)

agent = graph_builder.compile()


# ============ exceute agent ============

if __name__ == "__main__":
    result = agent.invoke({
        "topic": "go to picnic in the weekend",
        "level": "A2",
        "revision_count": 0,
        "max_revisions": 2
    })

    print("=" * 50)
    print("Final Article :")
    print(result["article"])
    print(f"\n revise time :{result['revision_count']}")
    print("\n vocabulary list:")
    for item in result["vocabulary"]:
        print(f"  {item['word']} — {item['meaning']}")
        print(f"    example:{item['example']}")

最终文章：
Last weekend, I went to the park with my friends. It was a sunny day, and we were very happy. We wanted to have a picnic. We packed a big bag with food and drinks. We took sandwiches, fruits, cookies, and juice. 

When we arrived at the park, we found a nice spot under a big tree. The grass was green and soft. We put a blanket on the grass. Then, we took out the food. The sandwiches were the first. They were tasty! We had cheese and ham in them. We also ate some apples and bananas. 

After eating, we played some games. We played frisbee and ran around. It was fun to be active outside. The weather was warm, and there were many people in the park. Some were walking dogs, and others were riding bikes. Everyone seemed to enjoy the day.

Later, we sat down and talked. We shared stories and laughed together. This made our picnic even more fun. After a few hours, we packed our things. We cleaned up the area and left the park happy.

Going to the park for a picnic is a great way to spend